# Dataset generation: TAL-NoiseMaker presets with CLAP embeddings

Renders random presets on Kaggle and stores parameters and CLAP audio embeddings in `.npz` shards.

## 0. Environment setup

In [ ]:
import subprocess, sys, os

def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        if check: raise RuntimeError(f"Command failed: {cmd}")
    return r

sh("apt-get -qq update && apt-get -qq install -y xvfb libxext6 libxrender1 libxcomposite1 "
   "libxcursor1 libxi6 libxtst6 libasound2 libfreetype6 libfontconfig1 2>/dev/null", check=False)

sh("pip -q install --no-deps laion-clap", check=False)
sh("pip -q install dawdreamer soundfile", check=False)
sh("pip -q install transformers torchlibrosa ftfy braceexpand webdataset wget", check=False)

print("\n[OK] Setup completed.")

## 1. Configuration

In [ ]:
import numpy as np, json, glob, time
from pathlib import Path

VST_PATH           = "/kaggle/input/tal-noisemaker-linux/TAL-NoiseMaker.vst3"
LOCKED_PARAMS_FILE = "/kaggle/input/locked-parameters/locked_parameters.json"
CLAP_CKPT          = "/kaggle/input/clap-music/music_audioset_epoch_15_esc_90.14.pt"

OUT_DIR  = "/kaggle/working"
PROVA_DIR = "/kaggle/working/_prova"

SAMPLE_RATE   = 48000
DURATION      = 4.0
NOTE_DURATION = 2.0
MIDI_NOTE     = 60
VELOCITY      = 100
MASTER_VOLUME = 0.8

RMS_THRESHOLD_MIN = 0.01
RMS_THRESHOLD_MAX = 0.85

CLAP_AUDIO_SR = 48000
CLAP_BATCH    = 128

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(PROVA_DIR).mkdir(parents=True, exist_ok=True)
print("[OK] Configuration loaded. SR =", SAMPLE_RATE)

## 2. Sampling priors and logical constraints

In [ ]:
_EPS = 0.05

CATEGORICAL_CHOICES = {
    "lfo1destination": np.linspace(0.0, 1.0, 8).tolist(),
    "lfo1waveform":    np.linspace(0.0, 1.0, 6).tolist(),
    "lfo1keytrigger":  [0.0, 1.0],
    "lfo2destination": np.linspace(0.0, 1.0, 8).tolist(),
    "lfo2waveform":    np.linspace(0.0, 1.0, 6).tolist(),
    "lfo2keytrigger":  [0.0, 1.0],
    "osc1waveform": [0.0, 0.5, 1.0],
    "osc1tune":     np.linspace(0.0, 1.0, 49).tolist(),
    "osc2waveform": np.linspace(0.0, 1.0, 5).tolist(),
    "osc2tune":     np.linspace(0.0, 1.0, 49).tolist(),
    "filtertype":        np.linspace(0.0, 1.0, 12).tolist(),
    "freeaddestination": np.linspace(0.0, 1.0, 6).tolist(),
    "oscsync":      [0.0, 1.0],
    "chorus1enable":[0.0, 1.0],
    "chorus2enable":[0.0, 1.0],
    "delaysync":    [0.0, 1.0],
    "delayx2l":     [0.0, 1.0],
    "delayx2r":     [0.0, 1.0],
}

BIPOLAR_NEUTRAL = {
    "lfo1amount", "lfo2amount", "osc1finetune", "osc2finetune",
    "osc1pw", "filtercontour", "freeadamount", "masterdetune",
}
P_BIPOLAR_NEUTRAL = 0.25
P_REVERB_OFF = 0.35
P_DELAY_OFF  = 0.35
P_OSC_OFF    = 0.10

def sample_param(p_name):
    if p_name in CATEGORICAL_CHOICES:
        return round(float(np.random.choice(CATEGORICAL_CHOICES[p_name])), 7)
    if p_name in BIPOLAR_NEUTRAL and np.random.random() < P_BIPOLAR_NEUTRAL:
        return 0.5
    if p_name == "reverbwet" and np.random.random() < P_REVERB_OFF:
        return 0.0
    if p_name == "delaywet" and np.random.random() < P_DELAY_OFF:
        return 0.0
    if p_name in ("osc1volume", "osc2volume") and np.random.random() < P_OSC_OFF:
        return 0.0
    return round(float(np.random.uniform(0.0, 1.0)), 7)

def enforce_audio_logic(params):
    def g(key, default=0.0):
        return float(params.get(key, default))

    if g("lfo1destination") == 0.0:
        for p in ["lfo1waveform", "lfo1rate", "lfo1phase", "lfo1keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo1amount" in params: params["lfo1amount"] = 0.5
    if abs(g("lfo1amount") - 0.5) < _EPS:
        for p in ["lfo1waveform", "lfo1rate", "lfo1phase", "lfo1keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo1destination" in params: params["lfo1destination"] = 0.0

    if g("lfo2destination") == 0.0:
        for p in ["lfo2waveform", "lfo2rate", "lfo2phase", "lfo2keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo2amount" in params: params["lfo2amount"] = 0.5
    if abs(g("lfo2amount") - 0.5) < _EPS:
        for p in ["lfo2waveform", "lfo2rate", "lfo2phase", "lfo2keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo2destination" in params: params["lfo2destination"] = 0.0

    if g("osc1waveform") != 0.5:
        if "osc1pw" in params: params["osc1pw"] = 0.5
    if g("osc2volume") < _EPS and g("osc2fm") < _EPS:
        if "oscsync" in params: params["oscsync"] = 0.0
    if g("osc1volume") < _EPS and g("osc2volume") < _EPS:
        if "ringmodulation" in params: params["ringmodulation"] = 0.0
        if "oscsync" in params:        params["oscsync"]        = 0.0

    if g("filtertype") == 1.0:
        for p in ["filtercutoff", "filterresonance",
                  "filterattack", "filterdecay", "filtersustain", "filterrelease",
                  "filterdrive"]:
            if p in params: params[p] = 0.0
        if "filtercontour" in params: params["filtercontour"] = 0.5
    if abs(g("filtercontour") - 0.5) < _EPS:
        for p in ["filterattack", "filterdecay", "filtersustain", "filterrelease"]:
            if p in params: params[p] = 0.0

    if g("freeaddestination") == 0.0:
        for p in ["freeadattack", "freeaddecay"]:
            if p in params: params[p] = 0.0
        if "freeadamount" in params: params["freeadamount"] = 0.5
    if abs(g("freeadamount") - 0.5) < _EPS:
        for p in ["freeadattack", "freeaddecay"]:
            if p in params: params[p] = 0.0
        if "freeaddestination" in params: params["freeaddestination"] = 0.0

    if g("reverbwet") < _EPS:
        for p in ["reverbdecay", "reverbpredelay", "reverbhighcut", "reverblowcut"]:
            if p in params: params[p] = 0.0
    if g("delaywet") < _EPS:
        for p in ["delaytime", "delaysync", "delayhighshelf", "delaylowshelf",
                  "delayfeedback", "delayx2l", "delayx2r"]:
            if p in params: params[p] = 0.0

    return params

def normalize_name(text):
    return text.lower().replace("_", "").replace(" ", "")

with open(LOCKED_PARAMS_FILE) as f:
    _locked = json.load(f)
WHITELIST = []
for category in _locked.values():
    WHITELIST.extend([normalize_name(p) for p in category])
PARAM_NAMES = list(WHITELIST)
print(f"[OK] {len(PARAM_NAMES)} parameters in the whitelist (column order of `params`).")

## 3. Headless rendering engine (Xvfb)

In [ ]:
import atexit, signal

def start_xvfb(display=":99"):
    if os.environ.get("_XVFB_UP") == display:
        return
    proc = subprocess.Popen(["Xvfb", display, "-screen", "0", "1280x720x24", "-ac"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2.0)
    os.environ["DISPLAY"] = display
    os.environ["_XVFB_UP"] = display
    atexit.register(lambda: proc.terminate())
    print(f"[OK] Xvfb running on {display}")

start_xvfb()

import dawdreamer as daw
import soundfile as sf

_engine = None
_synth  = None
_active = None
_volume_idx = None

def init_engine():
    global _engine, _synth, _active, _volume_idx
    if _engine is not None:
        return
    _engine = daw.RenderEngine(SAMPLE_RATE, 512)
    _synth  = _engine.make_plugin_processor("TAL-NoiseMaker", VST_PATH)
    desc = _synth.get_plugin_parameters_description()
    name_to_index = {normalize_name(p["name"]): p["index"] for p in desc}
    _active = [(name_to_index[n], n) for n in WHITELIST if n in name_to_index]
    missing = [n for n in WHITELIST if n not in name_to_index]
    if missing:
        print(f"[WARN] parameters not found in the VST: {missing}")
    _volume_idx = name_to_index.get("volume", None)
    print(f"[OK] VST loaded: {len(_active)} controllable parameters.")

def render_one(params: dict):
    init_engine()
    if _volume_idx is not None:
        _synth.set_parameter(_volume_idx, MASTER_VOLUME)
    for idx, name in _active:
        _synth.set_parameter(idx, float(params[name]))
    _synth.clear_midi()
    _synth.add_midi_note(MIDI_NOTE, VELOCITY, 0, NOTE_DURATION)
    _engine.load_graph([(_synth, [])])
    _engine.render(DURATION)
    audio = _engine.get_audio()
    mono  = audio.mean(axis=0).astype(np.float32)
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if not (RMS_THRESHOLD_MIN <= rms <= RMS_THRESHOLD_MAX):
        return None, audio
    return mono, audio

def sample_one_preset():
    cfg = {name: sample_param(name) for name in WHITELIST}
    cfg = enforce_audio_logic(cfg)
    return cfg

print("[OK] Rendering functions ready.")

## 4. CLAP encoder

In [ ]:
import torch
import laion_clap

_clap = None

def load_clap():
    global _clap
    if _clap is not None:
        return _clap
    dev = "cuda:0" if torch.cuda.is_available() else "cpu"
    _clap = laion_clap.CLAP_Module(enable_fusion=False, amodel="HTSAT-base", device=dev)
    if os.path.isfile(CLAP_CKPT):
        _clap.load_ckpt(CLAP_CKPT)
        print(f"[OK] CLAP from local checkpoint: {CLAP_CKPT}")
    else:
        _clap.load_ckpt()
        print("[WARN] local checkpoint not found: downloading the default LAION-CLAP checkpoint")
    return _clap

@torch.no_grad()
def clap_embed_batch(audio_list):
    clap = load_clap()
    L = int(round(DURATION * SAMPLE_RATE))
    arr = np.zeros((len(audio_list), L), np.float32)
    for i, a in enumerate(audio_list):
        n = min(len(a), L); arr[i, :n] = a[:n]
    x = torch.from_numpy(arr).float().to(next(clap.model.parameters()).device)
    emb = clap.get_audio_embedding_from_data(x=x, use_tensor=True)
    emb = emb.detach().cpu().numpy().astype(np.float32)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb = emb / np.clip(norms, 1e-8, None)
    return emb

print("[OK] CLAP encoder ready (loaded on first use).")

## B. Production: generating the `.npz` shards

In [ ]:
N_BATCH            = 10
CAMPIONI_PER_BATCH = 20000
START_BATCH        = 0
SEED_BASE          = 1234

def genera_batch(batch_id, n_target):
    t0 = time.time()
    init_engine()
    params_rows, audio_buf, emb_chunks, params_for_emb = [], [], [], []
    kept = attempts = 0

    def flush_emb():
        if not audio_buf: return
        emb_chunks.append(clap_embed_batch(audio_buf))
        audio_buf.clear()

    while kept < n_target:
        attempts += 1
        cfg = sample_one_preset()
        mono, _ = render_one(cfg)
        if mono is None:
            continue
        params_rows.append([cfg[name] for name in PARAM_NAMES])
        audio_buf.append(mono)
        kept += 1
        if len(audio_buf) >= CLAP_BATCH:
            flush_emb()
        if kept % 1000 == 0:
            el = time.time() - t0
            rate = kept / el
            eta = (n_target - kept) / max(rate, 1e-6)
            print(f"  batch {batch_id}: {kept}/{n_target}  "
                  f"({rate:.1f}/s, scarto {1-kept/attempts:.1%}, ETA {eta/60:.1f} min)")
    flush_emb()

    emb = np.concatenate(emb_chunks, axis=0).astype(np.float32)
    P   = np.array(params_rows, np.float32)
    src = np.zeros(len(P), np.int64)
    assert len(emb) == len(P) == n_target, f"mismatch: emb={len(emb)} P={len(P)}"

    out = os.path.join(OUT_DIR, f"shard_{batch_id:04d}.npz")
    np.savez(out, clap=emb, params=P, source=src)
    nm = np.linalg.norm(emb, axis=1).mean()
    print(f"[OK] shard {batch_id:04d}: N={n_target}  clap{emb.shape}  params{P.shape}  "
          f"|emb|≈{nm:.3f}  → {out}  ({os.path.getsize(out)/1e6:.1f} MB, "
          f"{(time.time()-t0)/60:.1f} min)")
    return out

def genera_produzione():
    init_engine(); load_clap()
    with open(os.path.join(OUT_DIR, "param_names.json"), "w") as fh:
        json.dump(PARAM_NAMES, fh)
    outs = []
    for k in range(START_BATCH, START_BATCH + N_BATCH):
        np.random.seed(SEED_BASE + k)
        outs.append(genera_batch(k, CAMPIONI_PER_BATCH))
    print(f"\n[OK] Production completed: {len(outs)} shards in {OUT_DIR}")
    print("     → use 'Save Version' to publish them as notebook output.")
    return outs

genera_produzione()
print("Ready. Set the parameters above and call genera_produzione() to start.")

## C. Checking a generated shard

In [ ]:
def verifica_shard(path):
    d = np.load(path)
    clap, params = d["clap"], d["params"]
    print(f"── {os.path.basename(path)} ──")
    print(f"  clap:   {clap.shape}  mean norm={np.linalg.norm(clap,axis=1).mean():.4f}  "
          f"NaN={np.isnan(clap).any()}")
    print(f"  params: {params.shape}  min={params.min():.3f} max={params.max():.3f}  "
          f"NaN={np.isnan(params).any()}")
    print(f"  source: {d['source'].shape}  unique values={np.unique(d['source'])}")
    ok = (clap.shape[1] == 512 and not np.isnan(clap).any()
          and params.min() >= 0 and params.max() <= 1)
    print(f"  → {'OK' if ok else 'PROBLEM'}")
    return ok